# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Finding 1: The brand-stature visibility ladder (§6.1)
The paper reports a large, statistically significant gap in unbranded AI visibility by brand tier: Tier 1 brands appear in 73% of unbranded category answers on their first run, Tier 2 in 44%, Tier 3 in just 11% — each step costing roughly 30 percentage points, with large effect sizes (Cohen's d 1.3-2.3).

My methodology question: Where does the "label" (brand tier) come from, and does the validation design carry the causal-sounding conclusion some readers might draw? The paper is careful to call this "cross-sectional and observational," which I respect — but the tier assignment itself comes from a hand-coded rubric (Wikipedia coverage, funding stage, press mentions), not a metric independent of the outcome being measured. Since stature indicators (funding, press, Wikipedia presence) likely correlate with the same underlying factors that drive AI training-data representation, I'd ask: how much of the "stature effect" is actually "web-presence volume effect" wearing a different name? The paper's own related-work section mentions the Matthew effect (LLMs prefer already-heavily-cited sources) — this suggests stature and citation-volume may be too entangled to cleanly separate as cause and effect, even with a well-designed observational study.

Finding 2: Persistence — mostly flat, with mild decline on ChatGPT/Perplexity (§6.4)
The paper finds most brands show flat visibility trajectories over repeated tracking runs, with ChatGPT and Perplexity showing a small but statistically real downward drift (CI excludes zero).

My methodology question: Does the validation design account for prompt-set drift as an alternative explanation? The paper itself raises this caveat honestly ("some of it may be prompt-set drift rather than true decay"), which I appreciate — but I'd push further: if the underlying prompt-generation process (seeded by "30-day market-research context") shifts what's being asked over time, then a "declining visibility" signal could really be a "changing question" signal. This is structurally similar to the "unbalanced history" issue I flagged in my own ML-04 data contract — when the measurement window itself isn't held perfectly constant, a real trend and a data artifact can look identical in the aggregate numbers.

Tone note: both papers/findings are asked in the spirit the paper itself invites — it's transparent about being observational and flags its own caveats, so my questions build on that honesty rather than contradict it.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
SELECT
    client_hash_id, content_hash_id, gsc_impressions,
    gsc_clicks * 1.0 / gsc_impressions as ctr,
    gsc_sum_position * 1.0 / gsc_impressions as avg_position
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
""").df()

df['log_impressions'] = np.log1p(df['gsc_impressions'])
df['position_tier_ctr'] = df['avg_position'].apply(lambda p: 0.0039 if p <= 10 else 0.0018)
df['underperforming'] = (df['ctr'] < df['position_tier_ctr']).astype(int)
features = ['log_impressions', 'avg_position']

def get_metrics(y_true, y_pred, name):
    return {'split': name, 'precision': precision_score(y_true, y_pred),
            'recall': recall_score(y_true, y_pred), 'f1': f1_score(y_true, y_pred)}

# BEFORE: naive random row-level split (no grouping)
X_tr, X_te, y_tr, y_te = train_test_split(df[features], df['underperforming'], test_size=0.2, random_state=42)
tree_naive = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42).fit(X_tr, y_tr)
naive_result = get_metrics(y_te, tree_naive.predict(X_te), 'BEFORE: naive random split')

# AFTER: grouped by client (same as ML-08)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))
train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]
tree_grouped = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42).fit(train_df[features], train_df['underperforming'])
grouped_result = get_metrics(test_df['underperforming'], tree_grouped.predict(test_df[features]), 'AFTER: grouped by client')

comparison = pd.DataFrame([naive_result, grouped_result])
comparison


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,split,precision,recall,f1
0,BEFORE: naive random split,0.975474,0.731866,0.836290
1,AFTER: grouped by client,0.977980,0.702657,0.817767


Before/after — naive vs. grouped split:

BEFORE (naive random split): precision=0.975, recall=0.732, f1=0.836
AFTER (grouped by client): precision=0.978, recall=0.703, f1=0.818

The gap here is small — only about 0.018 difference in f1, with the naive split actually scoring marginally higher (as expected in principle, since it can leak client-specific patterns). This is a smaller effect than I expected going in, and it's worth reporting honestly rather than dramatizing.

A likely explanation: my features (log_impressions, avg_position) are relatively generic, page-level signals rather than highly client-specific patterns — so a random split doesn't leak as much client identity as it might for a model using client-specific features (e.g., a client-embedding or client-average baseline). This doesn't mean grouped validation was unnecessary — it's still the methodologically correct choice, and I'd expect the gap to widen with richer, more client-specific features. But I'm reporting the actual, modest difference rather than overstating a lesson my data didn't clearly teach me this time.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Leakage audit on final feature set: log_impressions, avg_position
# Attack 1: does adding ctr directly (circular with label) inflate the score?
df['ctr_leak'] = df['ctr']
X_leak = df[features + ['ctr_leak']]
X_tr, X_te, y_tr, y_te = train_test_split(X_leak, df['underperforming'], test_size=0.2, random_state=42)
tree_leak = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42).fit(X_tr, y_tr)
leak_score = f1_score(y_te, tree_leak.predict(X_te))

# Confirm final feature set has NO future-window or product-flag columns
print(f"Final features used: {features}")
print(f"F1 with leaky ctr column added: {leak_score:.3f}")
print(f"F1 without leaky column (honest, from Section 2 AFTER): 0.818")
print(f"Leakage gap: {leak_score - 0.818:.3f}")


Final features used: ['log_impressions', 'avg_position']
F1 with leaky ctr column added: 1.000
F1 without leaky column (honest, from Section 2 AFTER): 0.818
Leakage gap: 0.182


Leakage audit results:

Adding ctr directly as a feature pushes F1 to 1.000 — a perfect score, confirming it's pure leakage (ctr is literally used to compute the label, underperforming = ctr < position_tier_ctr). This is a 0.182 leakage gap versus my honest score of 0.818.

Confirmed: my final feature set (log_impressions, avg_position) contains no label-derived columns, no future-window data, and no client-provided product flags. Both features are point-in-time, same-day GSC metrics available at the decision moment — consistent with the "available when" checks I performed in ML-05's feature vector notes. The leakage audit here reconfirms that finding on my final model's exact feature set, not just the earlier exploratory one.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My boldest original claim (from ML-08): "I trained a model that beat a hand-written business rule by 3x." (used in my Week 1 LinkedIn post, based on the original starter-notebook demo numbers)

Rewritten in safe language: In an initial exploratory comparison using the small anonymized starter dataset, a learned decision tree model showed a measured improvement over a simple hand-written rule on a Precision@50 metric, with the model scoring roughly 3x higher in that specific run. This was observed on a single sample and metric, and should be read as directional and decision-support — not a guarantee that any model will outperform any rule by this margin on other data, other metrics, or the full warehouse dataset. My later work (ML-08) on the real warehouse data showed a more nuanced picture: my baseline actually outperformed my trained model on the same task, once I accounted for the fact that my baseline and label were too tightly coupled — a reminder that "beats the baseline" claims need to be checked for exactly this kind of measurement artifact before being repeated as fact.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.